# SISPS-EXP-SOH Depth of Discharge

Full Depth of Discharge curves.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from concurrent import futures

from typing import List, Dict, Tuple

import darkdetect

if darkdetect.isDark():
    plt.style.use("dark_background")
    
plt.rcParams.update({
	'figure.dpi': 300,
	'figure.figsize': (8, 5),
})

%config InlineBackend.figure_formats = ['svg']

## Data Import

In [2]:
DATA_DIR = "../Data/"
PLOTS_DIR = "../Plots/"


def find_dod_files(data_dir) -> List[str]:
    """
    Find all Depth of Data Files within the Data/ directory based folder structure and file naming conventions.
    """
    files = []

    for root, dirs, dir_files in os.walk(data_dir):
        if root.split("/")[-1] != "DoD":
            continue

        for file in dir_files:
            if "UCT003-DOD-C" in file and file.endswith(".csv"):
                files.append(os.path.join(root, file))

    files.sort()

    return files


def import_dod_data(data_dir: str) -> List[pd.DataFrame]:
    # Filter for Depth of Data Files within Data/
    dod_files = find_dod_files(data_dir)

    dod_data = []
    HEADER_ROW = 19

    for file in dod_files:
        df = pd.read_csv(
            file,
            skiprows=HEADER_ROW - 1,
            skipfooter=1,
            usecols=[
                "Total Time",
                "Cycle",
                "Loop Counter #1",
                "Loop Counter #2",
                "Loop Counter #3",
                "Step",
                "Step time",
                "Current",
                "Voltage",
                "Power",
                "Amp-Hours",
                "Mode",
                "Data Acquisition Flag",
            ],
            dtype={
                "Total Time": float,
                "Cycle": int,
                "Loop Counter #1": int,
                "Loop Counter #2": int,
                "Loop Counter #3": int,
                "Step": int,
                "Step time": float,
                "Current": float,
                "Voltage": float,
                "Power": float,
                "Amp-Hours": float,
                "Mode": str,
                "Data Acquisition Flag": object,
            },
            engine="python",
        )

        df.attrs["test"] = file.split("/")[-1].split(".")[0]
        df.attrs["battery"] = file.split("/")[-1].split("-")[2]
        df.attrs["stage"] = file.split("/")[-1].split(".")[0].split("-")[3]

        dod_data.append(df)

    return dod_data


dod_data = import_dod_data(DATA_DIR)

## Depth of Discharge Values

In [3]:
for df in dod_data:
    dod = df["Amp-Hours"].min()
    print(f"{df.attrs['test']}: {dod:.2f} Ah")

UCT003-DOD-C01-S00-2: -6.36 Ah
UCT003-DOD-C02-S00-2: -6.96 Ah
UCT003-DOD-C03-S00-2: -6.88 Ah
UCT003-DOD-C04-S00-2: -7.06 Ah
UCT003-DOD-C05-S00-2: -6.91 Ah
UCT003-DOD-C06-S00-2: -6.27 Ah
UCT003-DOD-C07-S00-2: -7.07 Ah
UCT003-DOD-C08-S00: -2.33 Ah
UCT003-DOD-C09-S00: -2.29 Ah
UCT003-DOD-C10-S00: -3.10 Ah
UCT003-DOD-C11-S00: -2.99 Ah
UCT003-DOD-C12-S00: -2.95 Ah
UCT003-DOD-C13-S00: -2.85 Ah
UCT003-DOD-C14-S00: -2.79 Ah


## Basic Plotting

In [4]:
def plot_current_voltage(df: pd.DataFrame) -> plt.Figure:
    fig, ax = plt.subplots()
    ax2 = ax.twinx()

    ax.plot(df["Total Time"], df["Current"], label="Current")
    ax2.plot(df["Total Time"], df["Voltage"], color="r", label="Voltage")

    ax2.set_ylabel("Voltage (V)")
    ax.set_ylabel("Current (A)")

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2)

    return fig


def plot_all_dod_curves(
    dod_data: List[pd.DataFrame], save_dir: str | None = None
) -> None:
    def plot_save_curve(df, save_dir: str | None = None) -> None:
        fig = plot_current_voltage(df)
        fig.suptitle(df.attrs["test"])

        if save_dir:
            fig.savefig(f"{save_dir}/{df.attrs['test']}.png")

        plt.close(fig)

    with futures.ThreadPoolExecutor() as executor:
        for df in dod_data:
            executor.submit(plot_save_curve, df, save_dir)


os.makedirs(PLOTS_DIR, exist_ok=True)
plot_all_dod_curves(dod_data, PLOTS_DIR)
plt.close("all")

## Comparison of Depth of Discharge Curves

In [5]:
def plot_dod_steps(
    dfs: List[pd.DataFrame],
) -> Tuple[plt.Figure, plt.Figure, plt.Figure]:

    time_formatter = plt.FuncFormatter(
        lambda x, _: f"{x//3600:02.0f}:{(x%3600)//60:02.0f}"
    )
    voltage_formatter = plt.FuncFormatter(lambda x, _: f"{x:.1f} V")
    current_formatter = plt.FuncFormatter(lambda x, _: f"{x:.2f} A")
    # ampHour_formatter = plt.FuncFormatter(lambda x, _: f"{x:.0f} Ah")

    def get_pre_dod_fig() -> Tuple[plt.Figure, Dict[str, plt.Axes]]:
        fig = plt.figure(
            figsize=(18, 10),
        )
        ax = fig.subplot_mosaic(
            mosaic=[
                ["pre_discharge_voltage", "pre_recharge_voltage"],
                ["pre_discharge_voltage", "pre_recharge_current"],
            ],
        )

        ax["pre_discharge_voltage"].set_title("Pre-Discharge")
        ax["pre_recharge_voltage"].set_title("Pre-Recharge")

        ax["pre_recharge_voltage"].sharex(ax["pre_recharge_current"])

        ax["pre_discharge_voltage"].xaxis.set_major_formatter(time_formatter)
        ax["pre_recharge_voltage"].xaxis.set_major_formatter(time_formatter)
        ax["pre_recharge_current"].xaxis.set_major_formatter(time_formatter)
        ax["pre_discharge_voltage"].yaxis.set_major_formatter(voltage_formatter)
        ax["pre_recharge_voltage"].yaxis.set_major_formatter(voltage_formatter)
        ax["pre_recharge_current"].yaxis.set_major_formatter(current_formatter)

        return (fig, ax)

    fig1, ax1 = get_pre_dod_fig()

    def plot_pre_step(ax, df_discharge, df_recharge) -> None:
        if df_discharge.empty:
            return

        ax["pre_discharge_voltage"].plot(
            df_discharge["Stage time"],
            df_discharge["Voltage"],
            label=f"{df_recharge.attrs['battery']} {-df_discharge['Amp-Hours'].min():.2f} Ah",
        )

        ax["pre_recharge_voltage"].plot(
            df_recharge["Stage time"],
            df_recharge["Voltage"],
            label=f"{df_recharge.attrs['battery']} {df_recharge['Voltage'].iat[-1]:.3f} V",
        )
        ax["pre_recharge_current"].plot(
            df_recharge["Stage time"],
            df_recharge["Current"],
            label=f"{df_recharge.attrs['battery']} {df_recharge['Amp-Hours'].max():.2f} Ah",
        )

        ax["pre_discharge_voltage"].legend(title="Capacity")
        ax["pre_recharge_voltage"].legend(title="Est. EMF")
        ax["pre_recharge_current"].legend(title="Charge Accept")

    def get_dod_fig() -> Tuple[plt.Figure, List[plt.Axes]]:
        fig, ax = plt.subplots(
            1,
            2,
            figsize=(18, 10),
            gridspec_kw={"width_ratios": [3, 1]},
            sharey=True,
        )

        ax[0].set_title("Full Discharge")
        ax[1].set_title("Open Circuit Rest")

        ax[0].xaxis.set_major_formatter(time_formatter)
        ax[1].xaxis.set_major_formatter(time_formatter)
        ax[0].yaxis.set_major_formatter(voltage_formatter)

        return (fig, ax)

    fig2, ax2 = get_dod_fig()

    def plot_discharge_step(ax, df_discharge, df_rest) -> None:
        if df_discharge.empty:
            return

        ax[0].plot(
            df_discharge["Stage time"],
            df_discharge["Voltage"],
            label=f"{df_discharge.attrs['battery']} {-df_discharge['Amp-Hours'].min():.2f} Ah",
        )
        ax[1].plot(
            df_rest["Stage time"],
            df_rest["Voltage"],
            label=f"{df_rest.attrs['battery']} {df_rest['Voltage'].max():.3f} V",
        )

        ax[0].legend(title="Capacity")
        ax[1].legend(title="Est. EMF")

    def get_post_dod_fig() -> Tuple[plt.Figure, List[plt.Axes]]:
        fig, ax = plt.subplots(
            2,
            1,
            figsize=(12, 10),
            sharex=True,
        )

        ax[0].set_title("Post-Recharge")

        ax[1].xaxis.set_major_formatter(time_formatter)
        ax[0].yaxis.set_major_formatter(voltage_formatter)
        ax[1].yaxis.set_major_formatter(current_formatter)

        return (fig, ax)

    fig3, ax3 = get_post_dod_fig()

    def plot_post_step(ax, df_recharge) -> None:
        if df_recharge.empty:
            return

        ax[0].plot(
            df_recharge["Stage time"],
            df_recharge["Voltage"],
            label=f"{df_recharge.attrs['battery']} {df_recharge['Voltage'].iat[-1]:.3f} V",
        )
        ax[1].plot(
            df_recharge["Stage time"],
            df_recharge["Current"],
            label=f"{df_recharge.attrs['battery']} {df_recharge['Amp-Hours'].max():.2f} Ah",
        )

        ax[0].legend(title="Est. EMF")
        ax[1].legend(title="Charge Accept")

    for df in dfs:
        df_pre_discharge = df[(df["Step"] <= 1) & (df["Loop Counter #1"] == 1)].copy()
        df_pre_recharge = df[
            (df["Step"] >= 3) & (df["Step"] <= 7) & (df["Loop Counter #1"] == 1)
        ].copy()
        df_full_discharge = df[(df["Step"] == 1) & (df["Loop Counter #1"] == 2)].copy()
        df_full_discharge_rest = df[
            (df["Step"] == 2) & (df["Loop Counter #1"] == 2)
        ].copy()
        df_post_recharge = df[(df["Step"] >= 3) & (df["Loop Counter #1"] == 2)].copy()

        for df_stage in [
            df_pre_discharge,
            df_pre_recharge,
            df_full_discharge,
            df_full_discharge_rest,
            df_post_recharge,
        ]:
            df_stage["Stage time"] = (
                df_stage["Total Time"] - df_stage["Total Time"].min()
            )

        plot_pre_step(ax1, df_pre_discharge, df_pre_recharge)
        plot_discharge_step(ax2, df_full_discharge, df_full_discharge_rest)
        plot_post_step(ax3, df_post_recharge)

    fig1.tight_layout()
    fig2.tight_layout()
    fig3.tight_layout()

    return (fig1, fig2, fig3)


# Group DataFrames by Stage
from itertools import groupby

dod_data_by_stage = [
    list(group) for key, group in groupby(dod_data, lambda x: x.attrs["stage"])
]

for dfs_stage in dod_data_by_stage:
    stage = dfs_stage[0].attrs["stage"]
    fig1, fig2, fig3 = plot_dod_steps(dfs_stage)

    fig1.savefig(f"{PLOTS_DIR}/UCT003-DOD-{stage}-PRE-DSCH-CRG.png")
    fig2.savefig(f"{PLOTS_DIR}/UCT003-DOD-{stage}-FULL-DSCH.png")
    fig3.savefig(f"{PLOTS_DIR}/UCT003-DOD-{stage}-POST-CRG.png")
    plt.close("all")